# Notebook per Pre-training Self-Supervised (SpectraMAE)

Questo notebook implementa il pre-training di un modello basato su Transformer (SpectraMAE) utilizzando un approccio di tipo Masked Autoencoder. L'obiettivo è imparare una rappresentazione robusta degli spettri Raman in modo non supervisionato, che potrà poi essere usata per il fine-tuning su task di classificazione.

**Struttura del Notebook:**
1.  **Setup**: Caricamento delle librerie, impostazione del seed e del path di progetto.
2.  **Configurazione**: Lettura del file `.yaml` per caricare tutti gli iperparametri dell'esperimento.
3.  **Caricamento Dati**: Caricamento del dataset di pre-training (solo spettri, senza etichette).
4.  **Costruzione Modello**: Utilizzo della factory per creare il modello `Spectra_MAE` con i parametri specificati nel config.
5.  **Training Engine**: Definizione della logica di addestramento specifica per l'autoencoder, che calcola la loss di ricostruzione solo sulle patch mascherate.
6.  **Esecuzione**: Avvio del ciclo di addestramento e salvataggio del modello pre-addestrato.
7.  **Visualizzazione**: Grafico della loss e visualizzazione di alcuni esempi di spettri originali, mascherati e ricostruiti per validare qualitativamente il modello.

In [ ]:
# --- CELLA 1: SETUP AMBIENTE E IMPORTAZIONI ---
import sys
import os
import yaml
import numpy as np
import torch
from torch.utils.data import TensorDataset, DataLoader
import matplotlib.pyplot as plt

# ========== GLOBAL SEED FOR REPRODUCIBILITY ==========
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
print(f'✅ Global seed set to {SEED} for reproducibility')
# =====================================================

PROJECT_PATH = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_PATH not in sys.path:
    sys.path.insert(0, PROJECT_PATH)
os.chdir(PROJECT_PATH)
print(f'Working dir: {os.getcwd()}')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'✅ Setup completato! Stiamo usando: {device}')

In [ ]:
# --- CELLA 2: LETTURA CONFIGURAZIONE E CREAZIONE SCONTRINO ---
import shutil

# Prende il config dalla shell, o default se lanciato a mano
config_file = os.environ.get('EXP_CONFIG_FILE', 'configs/pretraining/exp_01_smae_baseline.yaml')
assert os.path.exists(config_file), f'Config non trovato: {config_file}'

with open(config_file, 'r') as f:
    config = yaml.safe_load(f)

# Applica il seed specificato nel file di config, se presente
split_seed = config.get('split', {}).get('seed', None)
if split_seed is not None:
    SEED = int(split_seed)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(SEED)
        torch.cuda.manual_seed_all(SEED)
    print(f'✅ Seed da YAML attivo: {SEED}')

output_dir = config_file.replace('configs', 'experiments').replace('.yaml', '')
os.makedirs(output_dir, exist_ok=True)
shutil.copy(config_file, os.path.join(output_dir, 'config_usato.yaml'))

print(f"Esperimento in esecuzione: {config.get('experiment_name', 'unknown')}")
print(f'Config usato: {config_file}')
print(f'I risultati verranno salvati in:\n{output_dir}')

In [ ]:
# --- CELLA 3: CARICAMENTO DATI ---
from src.data.mat_loader import load_mat_dataset
from sklearn.model_selection import train_test_split

ds_cfg = config.get('dataset', {})
dataset_path = ds_cfg.get('path')
print(f'Caricamento dati da: {dataset_path}...')

# Per il pre-training, carichiamo solo gli spettri (X)
loaded = load_mat_dataset(dataset_path, x_key=ds_cfg.get('x_key', 'X'))
X_data = loaded.X
N, L = X_data.shape
print(f"✅ Dati caricati: {N} spettri di lunghezza {L}")

# Split dei dati in train e validation
split_cfg = config.get('split', {}).get('holdout', {})
test_size = float(split_cfg.get('test_size', 0.2))
val_size_of_temp = float(split_cfg.get('val_size_of_temp', 0.25)) # 20% test, 20% val, 60% train

idx = np.arange(N)
idx_train, idx_temp = train_test_split(idx, test_size=test_size, random_state=SEED)
idx_val, idx_test = train_test_split(idx_temp, test_size=val_size_of_temp, random_state=SEED)

print(f'Pre-training | Train: {len(idx_train)}, Val: {len(idx_val)}, Test (unused): {len(idx_test)}')

BATCH_SIZE = int(ds_cfg.get('batch_size', 128))
train_loader = DataLoader(TensorDataset(torch.from_numpy(X_data[idx_train]).unsqueeze(1).float()), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(TensorDataset(torch.from_numpy(X_data[idx_val]).unsqueeze(1).float()), batch_size=BATCH_SIZE, shuffle=False)

In [ ]:
# --- CELLA 4: COSTRUZIONE MODELLO ---
from src.models.factory import build_model_from_config

# La factory ora supporta anche la creazione di Spectra_MAE
model = build_model_from_config(config, device)
print("✅ Modello Spectra_MAE costruito con successo.")
print(f"Numero di parametri: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

In [ ]:
# --- CELLA 5: TRAINING ENGINE PER MASKED AUTOENCODER ---
import torch.optim as optim
from tqdm.auto import tqdm

def train_mae_one_epoch(model, loader, optimizer, device):
    model.train()
    total_loss = 0
    for batch in tqdm(loader, desc="Training Epoch", leave=False):
        spectra = batch[0].to(device)
        optimizer.zero_grad()
        loss, _, _ = model(spectra) # Il modello MAE restituisce (loss, y_pred, mask)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

def evaluate_mae(model, loader, device):
    model.eval()
    total_loss = 0
    with torch.no_grad():
        for batch in tqdm(loader, desc="Validation", leave=False):
            spectra = batch[0].to(device)
            loss, _, _ = model(spectra)
            total_loss += loss.item()
    return total_loss / len(loader)

def train_mae(model, train_loader, val_loader, config, device):
    train_cfg = config.get('training', {})
    lr = float(train_cfg.get('learning_rate', 1e-3))
    wd = float(train_cfg.get('weight_decay', 0.05))
    epochs = int(train_cfg.get('epochs', 100))
    patience = int(train_cfg.get('patience', 10))

    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    
    scheduler_cfg = train_cfg.get('scheduler', {})
    if scheduler_cfg.get('name') == 'CosineAnnealingLR':
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=int(scheduler_cfg.get('T_max', epochs)))
    else:
        scheduler = None

    history = {'train_loss': [], 'val_loss': []}
    best_loss = float('inf')
    epochs_no_improve = 0
    best_state = None

    for epoch in range(epochs):
        train_loss = train_mae_one_epoch(model, train_loader, optimizer, device)
        val_loss = evaluate_mae(model, val_loader, device)
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)

        if scheduler:
            scheduler.step()

        print(f"Epoch {epoch+1}/{epochs} -> Train Loss: {train_loss:.6f}, Val Loss: {val_loss:.6f}")

        if val_loss < best_loss:
            best_loss = val_loss
            epochs_no_improve = 0
            best_state = model.state_dict()
            print(f"✨ Nuovo best model trovato con validation loss: {best_loss:.6f}")
        else:
            epochs_no_improve += 1

        if epochs_no_improve >= patience:
            print(f"🛑 Early stopping dopo {patience} epoche senza miglioramenti.")
            break
            
    return best_state, history

In [ ]:
# --- CELLA 6: ESECUZIONE ADDESTRAMENTO ---
print("="*40)
print("=== FASE DI PRE-TRAINING: SPECTRA MAE ===")
print("="*40)

best_weights, history = train_mae(model, train_loader, val_loader, config, device)

# Salva il modello migliore
model_save_path = os.path.join(output_dir, "best_pretrained_model.pth")
torch.save(best_weights, model_save_path)
print(f"✅ Modello pre-addestrato salvato in: {model_save_path}")

# Salva la history
history_path = os.path.join(output_dir, "pretrain_history.json")
import json
with open(history_path, 'w') as f:
    json.dump(history, f)

In [ ]:
# --- CELLA 7: VISUALIZZAZIONE RISULTATI ---
report_dir = os.path.join(output_dir, "report")
os.makedirs(report_dir, exist_ok=True)

# Grafico della loss
plt.figure(figsize=(10, 5))
plt.plot(history['train_loss'], label='Train Loss')
plt.plot(history['val_loss'], label='Validation Loss')
plt.title('Curve di Loss durante il Pre-training')
plt.xlabel('Epoche')
plt.ylabel('Reconstruction Loss')
plt.legend()
plt.grid(True)
plt.savefig(os.path.join(report_dir, "fig_pretrain_loss.png"))
plt.show()

# Visualizzazione ricostruzioni
print("\n--- Visualizzazione Esempi di Ricostruzione ---")
model.load_state_dict(best_weights)
model.eval()

# Prendiamo un batch di validazione
val_spectra_batch = next(iter(val_loader))[0].to(device)

with torch.no_grad():
    loss, y_pred, mask = model(val_spectra_batch)

# Denormalizza le patch per la visualizzazione
reconstructed_spectra = model.unpatchify(y_pred).cpu().numpy()
original_spectra = val_spectra_batch.cpu().numpy()

# Calcola lo spettro mascherato
mask = mask.detach()
mask = mask.unsqueeze(-1).repeat(1, 1, model.patch_size) # (N, n_patches, patch_size)
mask = model.unpatchify(mask).cpu().numpy() # (N, 1, L)
masked_spectra = original_spectra * (1 - mask)

# Plotta 4 esempi
n_examples = min(4, len(original_spectra))
fig, axes = plt.subplots(n_examples, 1, figsize=(12, 3 * n_examples))
if n_examples == 1: axes = [axes] # Rendi iterabile se c'è un solo plot

for i in range(n_examples):
    ax = axes[i]
    # Spettro originale
    ax.plot(original_spectra[i].squeeze(), label='Originale', color='blue')
    # Spettro ricostruito (solo dove era mascherato)
    ax.plot(reconstructed_spectra[i].squeeze(), label='Ricostruito', color='red', linestyle='--')
    # Spettro visibile (non mascherato)
    visible_part = np.ma.masked_where(mask[i].squeeze() == 1, original_spectra[i].squeeze())
    ax.plot(visible_part, label='Visibile (Input)', color='green', linewidth=2.5)
    
    ax.set_title(f'Esempio di Ricostruzione #{i+1}')
    ax.legend()
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(report_dir, "fig_reconstruction_examples.png"))
plt.show()

print("✅ Analisi qualitativa completata.")